## 06 - API Demo: SinOdio/SansHaine Hate Speech Detection

This notebook demonstrates the interaction with the SinOdio/SansHaine REST API,
deployed on Google Cloud Run. It complements the interactive Swagger UI available at
https://sinodio-api-1064831214069.europe-west4.run.app/docs.

The API contains four endpoints:
- `GET /` —> general project info and links to endpoints
- `GET /health` —> liveness check
- `POST /predict` —> classify a single text
- `POST /predict-batch` —> classify a list of texts with summary statistics

Only the `requests` library is required, which is part of the standard Python environment.

In [1]:
# =========================== #
# 06_api_demo.ipynb           #
# API interaction demo        #
# SinOdio/SansHaine — Phase 1 #
# =========================== #

import requests
import pandas as pd

# -- API base URL -------------------------------------------------
BASE_URL = "https://sinodio-api-1064831214069.europe-west4.run.app"

print(f"API base URL: {BASE_URL}")

API base URL: https://sinodio-api-1064831214069.europe-west4.run.app


## 1. Health Check

Verify that the API is running and the model is loaded.

In [2]:
# Health check
response = requests.get(f"{BASE_URL}/health")

print(f"Status code : {response.status_code}")
print(f"Response    : {response.json()}")

Status code : 200
Response    : {'status': 'ok', 'model': 'xlm-roberta-base'}


## 2. Single Text Prediction

Classify a single text using the `POST /predict` endpoint.

In [3]:
# Single prediction
text = "Ojalá que todos los inmigrantes se vayan de aquí"

response = requests.post(
    f"{BASE_URL}/predict",
    json={"text": text}
)

result = response.json()

print(f"Text       : {result['text']}")
print(f"Label      : {result['label']}")
print(f"Confidence : {result['confidence']:.2%}")
print(f"Prob hate  : {result['prob_hate']:.2%}")
print(f"Prob no hate: {result['prob_no_hate']:.2%}")

Text       : Ojalá que todos los inmigrantes se vayan de aquí
Label      : hate_speech
Confidence : 90.45%
Prob hate  : 90.45%
Prob no hate: 9.55%


## 3. Batch Prediction

Classify multiple texts in a single request using `POST /predict-batch`.
The endpoint accepts up to 50 texts and returns individual predictions
plus a summary with total counts and hate speech percentage.

In [5]:
# Batch prediction
texts = [
    # Explicit hate speech
    "Ojalá que todos los inmigrantes se vayan de aquí",
    "Estos refugiados son una plaga para nuestro país",
    # Neutral / non-hate
    "Hoy ha sido un día tranquilo en el parque",
    "La diversidad cultural de mi barrio me permite practicar muchos idiomas",
    # Ideological hate speech against specific groups
    "Las mujeres no deberían tener derecho a votar",
    # Ambiguous / borderline
    "No entiendo por qué tenemos que acoger a tanta gente cuando hay españoles pasándola mal",
    # Twitter-style with mentions and hashtags
    "@USER menuda panda de inútiles, todos a su país #sinvergüenzas",
]

response = requests.post(
    f"{BASE_URL}/predict-batch",
    json={"texts": texts}
)

result = response.json()

# Summary
print("=" * 55)
print(f"  SUMMARY")
print("=" * 55)
print(f"  Total texts    : {result['total']}")
print(f"  Hate speech    : {result['hate_count']} ({result['hate_percent']}%)")
print(f"  No hate speech : {result['no_hate_count']}")
print("=" * 55)

  SUMMARY
  Total texts    : 7
  Hate speech    : 4 (57.1%)
  No hate speech : 3


In [6]:
# Individual results as a DataFrame
df_results = pd.DataFrame(result['results'])

# Format for display
df_display = df_results[['text', 'label', 'confidence', 'prob_hate', 'prob_no_hate']].copy()
df_display['text']       = df_display['text'].str[:55] + '...'
df_display['confidence'] = df_display['confidence'].apply(lambda x: f"{x:.2%}")
df_display['prob_hate']  = df_display['prob_hate'].apply(lambda x: f"{x:.2%}")
df_display['prob_no_hate'] = df_display['prob_no_hate'].apply(lambda x: f"{x:.2%}")

df_display

,text,label,confidence,prob_hate,prob_no_hate
0,Ojalá que todos los inmigrantes se vayan de aq...,hate_speech,90.45%,90.45%,9.55%
1,Estos refugiados son una plaga para nuestro pa...,hate_speech,54.95%,54.95%,45.05%
2,Hoy ha sido un día tranquilo en el parque...,no_hate_speech,99.82%,0.18%,99.82%
3,La diversidad cultural de mi barrio me permite...,no_hate_speech,95.83%,4.17%,95.83%
4,Las mujeres no deberían tener derecho a votar...,no_hate_speech,94.47%,5.53%,94.47%
5,No entiendo por qué tenemos que acoger a tanta...,hate_speech,82.00%,82.00%,18.00%
6,"@USER menuda panda de inútiles, todos a su paí...",hate_speech,99.22%,99.22%,0.78%


## 4. Qualitative Analysis

The results above illustrate both the strengths and limitations of the Phase 1 classifier:

**Strengths:**
- Explicit hate speech (insults, dehumanising language) is detected with high confidence
- The `@USER` mention is correctly stripped by the preprocessing pipeline before classification
- Neutral text mentioning immigration in a factual or positive context is correctly classified as non-hate

**Limitations:**
- Ideological hate speech ("Las mujeres no deberían tener derecho a votar") is misclassified
  as non-hate. This type of hate speech denies rights to a group without using explicit insults,
  and is likely underrepresented in the training corpus which is predominantly Twitter data.
- Borderline cases (ambiguous or implicitly negative framing) produce lower confidence scores,
  reflecting genuine ambiguity in the training annotations.

These limitations are addressed in the future work section and motivate the manually annotated
corpus planned for Phase 3.